# ⚙️ NanoMill: Planetary Ball Mill Multiphysics Simulation & 3D CAD Suite
### Designed for Mechanical, Thermal & Particle Simulation for Nanoparticle Synthesis

This Google Colab notebook provides an end-to-end engineering simulation suite for designing a **high-energy planetary ball mill** dedicated to synthesizing **nanoparticles** (<100 nm).

**Upgrades & Features Included:**
1. **Mechanical Kinematics & Planetary Gears:** Planetary velocity ratios, gear teeth configuration ($Z_{sun}, Z_{planet}$), G-force fields (>35G), and required motor wattage & torque.
2. **DEM Ball Collision Dynamics:** Grinding ball impact velocity, collision frequency, impact energy spectrum, and specific power input ($W/g$).
3. **Nanoparticle Comminution Kinetics:** Size reduction model from micro-scale ($45\,\mu\text{m}$) down to the nanoscale grind limit ($<25\text{ nm}$).
4. **Thermal & Heat Dissipation:** Inelastic collision heat generation, conduction through vial walls with **annular cooling fins**, forced convective cooling, and transient heating curves.
5. **Light Mode 3D CAD with Black Technical Edges:** Interactive Three.js WebGL viewer with **mechanical spur gears**, **cooling fins**, **rotational indicator arrows** on the motor shaft and jar lids, and **interactive live gear ratio testing** ($k = -1.0, -1.5, -2.0, -2.5, -3.0$).
6. **Automated Engineering Report Generation:** Generates comprehensive technical documentation, charts, and Bill of Materials (BOM).

## 1. Setup & Environment
Google Colab has `numpy`, `scipy`, and `matplotlib` pre-installed. Run this cell to ensure all modules are loaded.

In [ ]:
# Check environment
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import os
from IPython.display import display, HTML, Markdown

print(f"Numpy: {np.__version__} | Scipy: {sp.__version__}")
print("Environment initialized successfully!")

## 2. Define Simulation & Planetary Gear Parameters
Adjust any of the parameters below to match your machine design specifications:

In [ ]:
# --- KINEMATIC & GEAR PARAMETERS ---
SUN_RPM = 480.0              # Sun disc rotational speed (RPM)
GEAR_RATIO_K = -2.0          # Transmission ratio omega/Omega (negative = counter-rotation)
SUN_RADIUS_MM = 160.0        # Sun disk radius in mm (axis to vial center)
VIAL_RADIUS_MM = 45.0        # Vial internal radius in mm
VIAL_HEIGHT_MM = 90.0        # Vial internal height in mm
NUM_VIALS = 4                # Number of milling vials (typically 2 or 4)

# --- GRINDING MEDIA & POWDER ---
MEDIA_MATERIAL = "Zirconia (ZrO2 - YSZ)"  # Options: Zirconia, Tungsten Carbide, Stainless Steel
BALL_DIAMETER_MM = 8.0       # Diameter of each grinding ball in mm
NUM_BALLS_PER_VIAL = 50      # Number of grinding balls per vial
POWDER_MASS_G = 25.0         # Powder charge per vial in grams

# --- COMMINUTION & THERMAL ---
INITIAL_D50_UM = 45.0        # Initial particle size in microns
MILLING_HOURS = 12.0         # Total milling run duration (hours)
USE_PCA_SURFACTANT = True    # Process Control Agent to prevent cold welding
VIAL_MATERIAL = "Stainless Steel (AISI 316)"  # Vial shell material
AMBIENT_TEMP_C = 25.0        # Ambient laboratory temperature

## 3. Run Multiphysics Simulation Engine
Execute the mechanical, DEM, comminution, and thermal solvers:

In [ ]:
import sys
sys.path.append(".")
from nanomill.kinematics import PlanetaryKinematics, MillGeometry, DriveSystemSizing
from nanomill.particles import BallMillDEM, NanoparticleKinetics
from nanomill.thermal import MillingThermalModel
from nanomill.cad_generator import BallMillCADGenerator
from nanomill.report import EngineeringReportGenerator

# 1. Kinematics
geom = MillGeometry(
    sun_radius_m=SUN_RADIUS_MM * 1e-3,
    vial_radius_m=VIAL_RADIUS_MM * 1e-3,
    vial_height_m=VIAL_HEIGHT_MM * 1e-3,
    num_vials=NUM_VIALS,
    vial_mass_kg=1.4,
)
kin = PlanetaryKinematics(geometry=geom, sun_rpm=SUN_RPM, gear_ratio_k=GEAR_RATIO_K)

# 2. Collision Dynamics
dem = BallMillDEM(
    kinematics=kin,
    media_material=MEDIA_MATERIAL,
    ball_diameter_mm=BALL_DIAMETER_MM,
    num_balls_per_vial=NUM_BALLS_PER_VIAL,
    powder_mass_per_vial_g=POWDER_MASS_G,
)
dyn = dem.calculate_collision_dynamics()

# 3. Comminution Kinetics
kinetics = NanoparticleKinetics(
    specific_power_w_per_g=dyn["specific_power_w_per_g"],
    initial_d50_um=INITIAL_D50_UM,
    use_surfactant_pca=USE_PCA_SURFACTANT,
)
kin_res = kinetics.simulate(milling_hours=MILLING_HOURS)

# 4. Thermal Model
thermal = MillingThermalModel(
    kinematics=kin,
    milling_power_per_vial_w=dyn["dissipated_impact_power_vial_w"],
    vial_material=VIAL_MATERIAL,
    ambient_temp_c=AMBIENT_TEMP_C,
)
therm_res = thermal.simulate_transient(total_milling_time_min=120.0)

# 5. Motor & Drive Sizing
drive = DriveSystemSizing(kinematics=kin)
drive_s = drive.calculate_motor_requirements(dyn["total_milling_power_all_vials_w"])

print("=== SIMULATION SUMMARY ===")
print(f"• Planetary Gear Ratio: k = {kin.gear_ratio_k:.2f} (Sun: {kin.gears.sun_teeth}T, Planet: {kin.gears.planet_teeth}T)")
print(f"• G-Force: {kin.g_force_sun():.1f} G ({kin.milling_regime()['regime']})")
print(f"• Ball Impact Velocity: {dyn['effective_impact_velocity_m_s']:.2f} m/s | Energy: {dyn['single_impact_energy_mj']:.2f} mJ")
print(f"• Predicted Final D50: {kin_res['final_d50_nm']:.1f} nm (Limit: {kin_res['d_limit_nm']:.0f} nm)")
print(f"• Equilibrium Temp: {therm_res['steady_state_temp_c']:.1f} °C (Max Safe: {therm_res['max_safe_temp_c']:.0f} °C)")
print(f"• Recommended Motor: {drive_s['recommended_motor_w']} W ({drive_s['recommended_motor_hp']} HP)")

## 4. Light Mode 3D Model Viewport with Black Edges & Gear Ratio Tester
The cell below renders the **Light Mode 3D CAD Viewport** directly inside Google Colab.
- **Black Technical CAD Edges** outlining all components
- **Mechanical Planetary Gears** (Sun Gear & Planet Gears with teeth)
- **Annular Cooling Fins** on all milling jars
- **Rotational Direction Indicators** on the central motor shaft and jar lids
- **Live Gear Ratio Tester buttons** (`k = -1.0`, `-1.5`, `-2.0`, `-2.5`, `-3.0`) allowing you to test speed ratios interactively in real time!

In [ ]:
cad = BallMillCADGenerator(kinematics=kin)
html_3d_code = cad.generate_interactive_html(width="100%", height="580px")
display(HTML(html_3d_code))

## 5. Visualizing Multiphysics Results
Plotting Ball Flight Paths, Nanoparticle Size Reduction, and Thermal Rise:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Ball Trajectories
traj = dem.simulate_sample_trajectories()
axes[0].plot(traj["vial_wall_x"] * 1e3, traj["vial_wall_y"] * 1e3, 'k-', lw=2.5, label="Vial Wall")
for idx, (tx, ty) in enumerate(traj["trajectories"]):
    axes[0].plot(tx * 1e3, ty * 1e3, lw=2.0, ls='--', label=f"Ball #{idx+1}")
axes[0].set_aspect('equal')
axes[0].set_title("Cataracting Ball Trajectories", fontweight="bold")
axes[0].set_xlabel("X (mm)")
axes[0].set_ylabel("Y (mm)")
axes[0].grid(True, linestyle=":", alpha=0.6)
axes[0].legend(loc="upper right", fontsize=8)

# 2. Nanoparticle Kinetics
axes[1].semilogy(kin_res["time_hours"], kin_res["d50_nm"], 'b-', lw=2.5, label="D50 Median")
axes[1].semilogy(kin_res["time_hours"], kin_res["d90_nm"], 'm--', lw=1.8, label="D90 Top Cut")
axes[1].axhline(100.0, color='r', ls=':', lw=1.5, label="Nano Limit (100nm)")
axes[1].axhline(kin_res["d_limit_nm"], color='g', ls='-.', label=f"Limit ({kin_res['d_limit_nm']:.0f}nm)")
axes[1].set_title("Nanoparticle Size vs Milling Time", fontweight="bold")
axes[1].set_xlabel("Time (hours)")
axes[1].set_ylabel("Particle Diameter (nm)")
axes[1].grid(True, which="both", linestyle=":", alpha=0.6)
axes[1].legend(fontsize=8)

# 3. Thermal Profile
axes[2].plot(therm_res["time_minutes"], therm_res["internal_temp_c"], 'r-', lw=2.5, label="Internal Temp")
axes[2].axhline(therm_res["steady_state_temp_c"], color='orange', ls=':', label=f"Eq ({therm_res['steady_state_temp_c']}°C)")
axes[2].axhline(therm_res["max_safe_temp_c"], color='darkred', ls='-.', label=f"Safe Limit ({therm_res['max_safe_temp_c']}°C)")
axes[2].set_title("Vial Thermal Equilibrium Curve", fontweight="bold")
axes[2].set_xlabel("Time (minutes)")
axes[2].set_ylabel("Temperature (°C)")
axes[2].grid(True, linestyle=":", alpha=0.6)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Export 3D Models & Generate Engineering Report
Export CAD files (`.scad`, `.stl`) with gears and cooling fins, and full markdown report.

In [ ]:
os.makedirs("output", exist_ok=True)

# Export 3D models
scad_path = cad.export_openscad_file("output/planetary_ball_mill_gears_fins.scad")
stl_path = cad.export_stl_file("output/planetary_ball_mill_gears_fins.stl")

# Export report
reporter = EngineeringReportGenerator(
    kinematics=kin,
    drive_sizing=drive,
    dem=dem,
    kinetics=kinetics,
    thermal=thermal,
)
md_report = reporter.generate_markdown_report()

print("Exported Files:")
print(f" [+] OpenSCAD Code: {scad_path}")
print(f" [+] Binary STL: {stl_path}")

display(Markdown(md_report))